<div style="font-size:2.3em;font-weight:600;margin-bottom:10px">Agentic Artificial Intelligence</div>
<div style="font-size:1.7em;margin-bottom:14px">Unit 10 · Multi-agent systems</div>

The last unit: today we build a **team of agents**.

**Multi-agent system:** several agents, each with its own job.

<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px 32px"><div><b>1</b>&nbsp; Opening: an office full of agents</div><div><b>5</b>&nbsp; Back to the supervisor</div><div><b>2</b>&nbsp; How a team can be organized</div><div><b>6</b>&nbsp; A round table: the network</div><div><b>3</b>&nbsp; A supervisor and three specialists</div><div><b>7</b>&nbsp; Do you need more than one agent?</div><div><b>4</b>&nbsp; Specialists as tools</div><div><b>8</b>&nbsp; Wrap-up of the unit and of the course</div></div>

<div style="height:44px"></div>

## Part 1 of 8 · Opening: an office full of agents

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What if one agent is not enough?**

#### 1.1 · One sentence for today

<details>
<summary>Click to show</summary>

**A multi-agent system is several agents, and often they are one model with different prompts. The hard part is not the agents. The hard part is: who acts next, and who knows what.**

</details>

#### 1.2 · Why a team, and what is the price?

**One agent** cannot be an expert in everything.

**One small agent** is easier to change than one giant prompt.

| | One agent | Several agents |
|:--|:--|:--|
| **Complexity** | simpler to build and debug | **more complex**, they must be coordinated |
| **Skills** | general | specialized |
| **Changes** | one big prompt | one small agent at a time |

<div style="height:44px"></div>

## Part 2 of 8 · How a team can be organized

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What are the ways to organize a team, and to pass on the work?**

#### 2.1 · Three architectures

```
 Supervisor            Network, or swarm         Hierarchical

  Manager                A <---> B                   Top
  /  |  \                 \     /                   /   \
 W   W   W                 \   /                 Mid     Mid
                             C                  / \     / \
                                               W   W   W   W
```

**W** is a worker, our word for a specialist.

**Supervisor:** one manager gives out all the work.

**Network:** no manager, like people at a round table.

**Hierarchical:** a big company, with managers of managers.

#### 2.2 · Two ways to pass on the work, and one word to note

| | Tool calling | Handoff |
|:--|:--|:--|
| **Like** | a manager who asks a specialist and gets a report | a hotline: “I will transfer you” |
| **Who talks to you?** | only the manager | the next person |
| **Use it when** | the agent only has to do a task | the agent must keep talking with the user |

**Shared state:** all agents read and write the same list of messages. So each agent can see everything.

**The price:** an agent that reads all of it pays for all of it in tokens.

<div style="height:44px"></div>

## Part 3 of 8 · A supervisor and three specialists

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How do we build this office in code?**

In [ ]:
# 3.1 · The parts that we need
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, Literal
from typing_extensions import TypedDict

load_dotenv()

print("✅ Environment setup complete!")

In [ ]:
# 3.2 · The state of our office, and the model
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")

print("✅ State schema and LLM initialized!")

In [ ]:
# 3.3 · The research specialist
def research_agent(state: AgentState) -> AgentState:
    messages = state["messages"]

    system_prompt = """You are a research specialist. Your role is to:
- Find and gather relevant information
- Summarize findings clearly
- Cite sources when possible
- Provide comprehensive research on topics

Be thorough and accurate in your research."""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    response = llm.invoke(messages)

    return {
        "messages": [response],
        "next_agent": "supervisor"
    }

#### 3.4 · What did we just see?

<details>
<summary>Click to show</summary>

**A job title in a prompt is not a skill.**

This agent has no search tool and no internet.

So if it cites a source, please check that the source exists.

</details>

In [ ]:
# 3.5 · The coding specialist
def coding_agent(state: AgentState) -> AgentState:
    messages = state["messages"]

    system_prompt = """You are a coding specialist. Your role is to:
- Write clean, well-documented code
- Explain code clearly
- Debug and fix issues
- Follow best practices

Always provide working code examples with explanations."""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    response = llm.invoke(messages)

    return {
        "messages": [response],
        "next_agent": "supervisor"
    }

In [ ]:
# 3.6 · The analysis specialist
def analysis_agent(state: AgentState) -> AgentState:
    messages = state["messages"]

    system_prompt = """You are an analysis specialist. Your role is to:
- Analyze data and information
- Provide logical reasoning
- Break down complex problems
- Draw conclusions from evidence

Be methodical and clear in your analysis."""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    response = llm.invoke(messages)

    return {
        "messages": [response],
        "next_agent": "supervisor"
    }

print("✅ Worker agents created!")

In [ ]:
# 3.7 · A form for the decision
from pydantic import BaseModel

class SupervisorDecision(BaseModel):
    """Schema for supervisor agent decision. Only allows one word: 'research', 'coding', 'analysis', or 'end'."""
    next_agent: Literal["research", "coding", "analysis", "end"]

In [ ]:
# 3.8 · The supervisor
def supervisor_agent(state: AgentState) -> AgentState:
    messages = state["messages"]

    last_message = messages[-1].text if messages else ""

    system_prompt = """You are a supervisor agent coordinating a team of specialists:
- research_agent: For research, and fact-finding tasks. Not for coding or analysis.
- coding_agent: For programming, code writing, debugging, and technical implementation
- analysis_agent: For data analysis, reasoning, problem-solving, and logical thinking

Based on the user's request, decide which agent should handle it. Respond with ONLY one word: 'research', 'coding', or 'analysis', or 'end' if the task is complete."""

    decision_prompt = f"""User request: {last_message}

Which agent should handle this? Respond with ONLY one word: 'research', 'coding', 'analysis', or 'end'."""

    decision_messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=decision_prompt)
    ]
    structured_decision_llm = llm.with_structured_output(SupervisorDecision)
    decision = structured_decision_llm.invoke(decision_messages)
    next_agent = decision.next_agent.strip().lower()
    print(f"Supervisor decided to route to: {next_agent}")

    if next_agent not in ["research", "coding", "analysis", "end"]:
        next_agent = "research"

    return {
        "messages": [],
        "next_agent": next_agent
    }

print("✅ Supervisor agent created!")

In [ ]:
# 3.9 · Who comes next?
def route_to_agent(state: AgentState) -> Literal[
    "research_agent", "coding_agent", "analysis_agent", "supervisor", "__end__"
]:
    next_agent = state.get("next_agent", "supervisor")

    if next_agent == "research":
        return "research_agent"
    elif next_agent == "coding":
        return "coding_agent"
    elif next_agent == "analysis":
        return "analysis_agent"
    elif next_agent == "end":
        return "__end__"
    else:
        return "supervisor"

In [ ]:
# 3.10 · The graph: four nodes, and a split in the road
workflow = StateGraph(AgentState)

workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("research_agent", research_agent)
workflow.add_node("coding_agent", coding_agent)
workflow.add_node("analysis_agent", analysis_agent)

workflow.set_entry_point("supervisor")

workflow.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "research_agent": "research_agent",
        "coding_agent": "coding_agent",
        "analysis_agent": "analysis_agent",
        "__end__": END,
        "supervisor": "supervisor"
    }
)

In [ ]:
# 3.11 · The edges after the specialists
workflow.add_edge("research_agent", END)
workflow.add_edge("coding_agent", END)
workflow.add_edge("analysis_agent", END)

app = workflow.compile()

print("✅ Multi-agent supervisor system built!")
print("\nGraph structure:")
print("START → supervisor → [research_agent | coding_agent | analysis_agent] "
      "→ supervisor → END")

In [ ]:
# 3.12 · Test 1
print("=" * 60)
print("TEST 1: Research Query")
print("=" * 60)

research_query = "What are the latest developments in large language models?"
initial_state = {
    "messages": [HumanMessage(content=research_query)],
    "next_agent": "supervisor"
}

result = app.invoke(initial_state)

for message in result["messages"]:
    print(type(message).__name__ + ":", message.text[:200])
print("next_agent:", result["next_agent"])

In [ ]:
# 3.13 · Test 2
print("\n" + "=" * 60)
print("TEST 2: Coding Query")
print("=" * 60)

coding_query = (
    "Write a Python function to calculate the factorial of a number "
    "with error handling."
)
initial_state = {
    "messages": [HumanMessage(content=coding_query)],
    "next_agent": "supervisor"
}

result = app.invoke(initial_state)

for message in result["messages"]:
    print(type(message).__name__ + ":", message.text[:200])
print("next_agent:", result["next_agent"])

In [ ]:
# 3.14 · Test 3
print("\n" + "=" * 60)
print("TEST 3: Analysis Query")
print("=" * 60)

analysis_query = (
    "Analyze the pros and cons of using microservices vs monolithic architecture "
    "for a startup."
)
initial_state = {
    "messages": [HumanMessage(content=analysis_query)],
    "next_agent": "supervisor"
}

result = app.invoke(initial_state)

for message in result["messages"]:
    print(type(message).__name__ + ":", message.text[:200])
print("next_agent:", result["next_agent"])

<div style="height:44px"></div>

## Part 4 of 8 · Specialists as tools

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What if the manager only wants a report?**

In [ ]:
# 4.1 · A state with only messages
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END

class CustomState(MessagesState):
    pass

In [ ]:
# 4.2 · The research specialist, as its own small graph
def create_research_subgraph():
    class ResearchState(MessagesState):
        pass

    def research_node(state: ResearchState):
        messages = state["messages"]
        system_prompt = """You are a research specialist. Your role is to:
- Find and gather relevant information
- Summarize findings clearly
- Cite sources when possible
- Provide comprehensive research on topics

Be thorough and accurate in your research."""

        if not any(isinstance(msg, SystemMessage) for msg in messages):
            messages = [SystemMessage(content=system_prompt)] + messages

        response = llm.invoke(messages)
        return {"messages": [response]}

    graph = StateGraph(ResearchState)
    graph.add_node("research", research_node)
    graph.set_entry_point("research")
    graph.add_edge("research", END)
    return graph.compile()

In [ ]:
# 4.3 · The coding specialist, the same way
def create_coding_subgraph():
    class CodingState(MessagesState):
        pass

    def coding_node(state: CodingState):
        messages = state["messages"]
        system_prompt = """You are a coding specialist. Your role is to:
- Write clean, well-documented code
- Explain code clearly
- Debug and fix issues
- Follow best practices

Always provide working code examples with explanations."""

        if not any(isinstance(msg, SystemMessage) for msg in messages):
            messages = [SystemMessage(content=system_prompt)] + messages

        response = llm.invoke(messages)
        return {"messages": [response]}

    graph = StateGraph(CodingState)
    graph.add_node("coding", coding_node)
    graph.set_entry_point("coding")
    graph.add_edge("coding", END)
    return graph.compile()

In [ ]:
# 4.4 · The analysis specialist, and all three graphs
def create_analysis_subgraph():
    class AnalysisState(MessagesState):
        pass

    def analysis_node(state: AnalysisState):
        messages = state["messages"]
        system_prompt = """You are an analysis specialist. Your role is to:
- Analyze data and information
- Provide logical reasoning
- Break down complex problems
- Draw conclusions from evidence

Be methodical and clear in your analysis."""

        if not any(isinstance(msg, SystemMessage) for msg in messages):
            messages = [SystemMessage(content=system_prompt)] + messages

        response = llm.invoke(messages)
        return {"messages": [response]}

    graph = StateGraph(AnalysisState)
    graph.add_node("analysis", analysis_node)
    graph.set_entry_point("analysis")
    graph.add_edge("analysis", END)
    return graph.compile()

research_subgraph = create_research_subgraph()
coding_subgraph = create_coding_subgraph()
analysis_subgraph = create_analysis_subgraph()

print("✅ Subagent graphs created!")

In [ ]:
# 4.5 · An agent becomes a tool
@tool(
    "research_agent",
    description=(
        "Use this tool for research, fact-finding, and information gathering tasks. "
        "Not for coding or analysis."
    )
)
def call_research_agent(query: str) -> str:
    subagent_input = {
        "messages": [HumanMessage(content=query)]
    }

    result = research_subgraph.invoke(subagent_input)

    return result["messages"][-1].text

In [ ]:
# 4.6 · The second tool
@tool(
    "coding_agent",
    description=(
        "Use this tool for programming, code writing, debugging, "
        "and technical implementation tasks."
    )
)
def call_coding_agent(query: str) -> str:
    subagent_input = {
        "messages": [HumanMessage(content=query)]
    }

    result = coding_subgraph.invoke(subagent_input)
    return result["messages"][-1].text

In [ ]:
# 4.7 · The third tool, and the list
@tool(
    "analysis_agent",
    description=(
        "Use this tool for data analysis, reasoning, problem-solving, "
        "and logical thinking tasks."
    )
)
def call_analysis_agent(query: str) -> str:
    subagent_input = {
        "messages": [HumanMessage(content=query)]
    }

    result = analysis_subgraph.invoke(subagent_input)
    return result["messages"][-1].text

tools = [call_research_agent, call_coding_agent, call_analysis_agent]

print("✅ Tools created for subagents!")

In [ ]:
# 4.8 · The supervisor, with a menu
def supervisor_node(state: CustomState) -> CustomState:
    messages = state["messages"]

    system_prompt = """You are a supervisor agent coordinating a team of specialists:
- research_agent: For research, fact-finding, and information gathering tasks. Not for coding or analysis.
- coding_agent: For programming, code writing, debugging, and technical implementation
- analysis_agent: For data analysis, reasoning, problem-solving, and logical thinking

When a user asks a question, decide which specialist tool to use and call it. 
After getting the result from the tool, synthesize the response for the user.
You can call multiple tools if needed to fully answer the user's question."""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    supervisor_llm = llm.bind_tools(tools)

    response = supervisor_llm.invoke(messages)

    return {"messages": [response]}

In [ ]:
# 4.9 · Two nodes, and a split in the road
tool_calling_workflow = StateGraph(CustomState)

tool_calling_workflow.add_node("supervisor", supervisor_node)
tool_calling_workflow.add_node("tools", ToolNode(tools))

tool_calling_workflow.set_entry_point("supervisor")

def should_continue(state: CustomState) -> Literal["tools", "end"]:
    messages = state["messages"]
    last_message = messages[-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "end"

tool_calling_workflow.add_conditional_edges(
    "supervisor",
    should_continue,
    {
        "tools": "tools",
        "end": END
    }
)

In [ ]:
# 4.10 · After the tools
tool_calling_workflow.add_edge("tools", "supervisor")

tool_calling_app = tool_calling_workflow.compile()

print("✅ Tool-calling multi-agent system built!")
print("\nGraph structure:")
print("START → supervisor → [tools (if needed)] → supervisor → END")

In [ ]:
# 4.11 · The research question again
print("=" * 60)
print("TEST: Tool-Calling Multi-Agent System")
print("=" * 60)

test_query = "What are the latest developments in large language models?"
initial_state = {
    "messages": [HumanMessage(content=test_query)]
}

result = tool_calling_app.invoke(initial_state)

print(f"\nUser Query: {test_query}")
print(f"\nNumber of messages: {len(result['messages'])}")
print(f"""\nNumber of tool calls: {len([
    msg for msg in result['messages'] if hasattr(msg, 'tool_calls') and msg.tool_calls
])}""")
print("\n" + "=" * 60)
print("FINAL RESPONSE:")
print("=" * 60)

for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print(f"\n{msg.text}")
    elif hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"\n[Tool calls made: {[tc['name'] for tc in msg.tool_calls]}]")

In [ ]:
# 4.12 · What did the supervisor write?
for message in result["messages"]:
    print(type(message).__name__)
    for call in getattr(message, "tool_calls", []):
        print("   tool_calls:", call["name"], call["args"])
    if message.text:
        print("   " + message.text[:200])

#### 4.13 · Our two versions, and a real handoff

| | First version | Tool calling |
|:--|:--|:--|
| **Who answers the user?** | the specialist | the supervisor |
| **What does the supervisor do?** | it only sends the work on | it asks, reads the report, and writes the answer |

**A real handoff:** the active agent itself decides to pass on the work, and the user then talks to the next agent.

Today we do not build that. We only learn the idea.

<div style="height:44px"></div>

## Part 5 of 8 · Back to the supervisor

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What if the work comes back to the supervisor?**

In [ ]:
# 5.1 · One state for all three small graphs
from langgraph.graph import MessagesState

class AgentSubgraphState(MessagesState):
    pass

In [ ]:
# 5.2 · The research subgraph, again
def create_research_subgraph():
    def research_node(state: AgentSubgraphState):
        messages = state["messages"]
        system_prompt = """You are a research specialist. Your role is to:
- Find and gather relevant information
- Summarize findings clearly
- Cite sources when possible
- Provide comprehensive research on topics

Be thorough and accurate in your research."""

        if not any(isinstance(msg, SystemMessage) for msg in messages):
            messages = [SystemMessage(content=system_prompt)] + messages

        response = llm.invoke(messages)
        return {"messages": [response]}

    graph = StateGraph(AgentSubgraphState)
    graph.add_node("research", research_node)
    graph.set_entry_point("research")
    graph.add_edge("research", END)

    return graph.compile()

In [ ]:
# 5.3 · The coding subgraph, again
def create_coding_subgraph():
    def coding_node(state: AgentSubgraphState):
        messages = state["messages"]
        system_prompt = """You are a coding specialist. Your role is to:
- Write clean, well-documented code
- Explain code clearly
- Debug and fix issues
- Follow best practices

Always provide working code examples with explanations."""

        if not any(isinstance(msg, SystemMessage) for msg in messages):
            messages = [SystemMessage(content=system_prompt)] + messages

        response = llm.invoke(messages)
        return {"messages": [response]}

    graph = StateGraph(AgentSubgraphState)
    graph.add_node("coding", coding_node)
    graph.set_entry_point("coding")
    graph.add_edge("coding", END)

    return graph.compile()

In [ ]:
# 5.4 · The analysis subgraph, and all three again
def create_analysis_subgraph():
    def analysis_node(state: AgentSubgraphState):
        messages = state["messages"]
        system_prompt = """You are an analysis specialist. Your role is to:
- Analyze data and information
- Provide logical reasoning
- Break down complex problems
- Draw conclusions from evidence

Be methodical and clear in your analysis."""

        if not any(isinstance(msg, SystemMessage) for msg in messages):
            messages = [SystemMessage(content=system_prompt)] + messages

        response = llm.invoke(messages)
        return {"messages": [response]}

    graph = StateGraph(AgentSubgraphState)
    graph.add_node("analysis", analysis_node)
    graph.set_entry_point("analysis")
    graph.add_edge("analysis", END)

    return graph.compile()

research_subgraph = create_research_subgraph()
coding_subgraph = create_coding_subgraph()
analysis_subgraph = create_analysis_subgraph()

print("✅ Agent subgraphs created!")

In [ ]:
# 5.5 · A small function around each subgraph
def invoke_research_subgraph(state: AgentState) -> AgentState:
    result = research_subgraph.invoke(state)
    return {
        "messages": result["messages"],
        "next_agent": "supervisor"
    }

def invoke_coding_subgraph(state: AgentState) -> AgentState:
    result = coding_subgraph.invoke(state)
    return {
        "messages": result["messages"],
        "next_agent": "supervisor"
    }

def invoke_analysis_subgraph(state: AgentState) -> AgentState:
    result = analysis_subgraph.invoke(state)
    return {
        "messages": result["messages"],
        "next_agent": "supervisor"
    }

In [ ]:
# 5.6 · The same shape as the first graph
supervisor_workflow = StateGraph(AgentState)

supervisor_workflow.add_node("supervisor", supervisor_agent)
supervisor_workflow.add_node("research_agent", invoke_research_subgraph)
supervisor_workflow.add_node("coding_agent", invoke_coding_subgraph)
supervisor_workflow.add_node("analysis_agent", invoke_analysis_subgraph)

supervisor_workflow.set_entry_point("supervisor")

supervisor_workflow.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "research_agent": "research_agent",
        "coding_agent": "coding_agent",
        "analysis_agent": "analysis_agent",
        "__end__": END,
        "supervisor": "supervisor"
    }
)

In [ ]:
# 5.7 · The edges after the specialists, this time
supervisor_workflow.add_edge("research_agent", "supervisor")
supervisor_workflow.add_edge("coding_agent", "supervisor")
supervisor_workflow.add_edge("analysis_agent", "supervisor")

supervisor_app = supervisor_workflow.compile()

print("✅ Supervisor system with subgraphs built!")

In [ ]:
# 5.8 · The coding question again
print("\n" + "=" * 60)
print("Coding Query")
print("=" * 60)

coding_query = (
    "Write a Python function to calculate the factorial of a number "
    "with error handling."
)
initial_state = {
    "messages": [HumanMessage(content=coding_query)],
    "next_agent": "supervisor"
}

result = supervisor_app.invoke(initial_state)

for message in result["messages"]:
    print(type(message).__name__ + ":", message.text[:200])
print("next_agent:", result["next_agent"])

#### 5.9 · What did we just see?

<details>
<summary>Click to show</summary>

**A loop needs a limit.**

**Recursion limit:** the highest number of steps in one run. Our version of LangGraph has about 10,000 by default.

So set your own:

```python
supervisor_app.invoke(initial_state, config={"recursion_limit": 10})
```

</details>

<div style="height:44px"></div>

## Part 6 of 8 · A round table: the network

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What if there is no manager?**

In [ ]:
# 6.1 · The state, with a counter
class NetworkState(TypedDict):
    messages: Annotated[list, add_messages]
    current_agent: str
    round_count: int

In [ ]:
# 6.2 · Agent A
def agent_a(state: NetworkState) -> NetworkState:
    messages = state["messages"]
    round_count = state.get("round_count", 0)

    system_prompt = """You are Agent A, a research specialist. 
You gather information and share findings with other agents.
When you receive input, research the topic and share your findings.
If another agent has already responded, build upon their work."""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    context = f"This is round {round_count + 1} of the network conversation."
    messages.append(HumanMessage(content=context))

    response = llm.invoke(messages)

    return {
        "messages": [response],
        "current_agent": "agent_b",
        "round_count": round_count + 1
    }

In [ ]:
# 6.3 · Agent B
def agent_b(state: NetworkState) -> NetworkState:
    messages = state["messages"]
    round_count = state.get("round_count", 0)

    system_prompt = """You are Agent B, an analysis specialist.
You analyze information provided by other agents and add insights.
Build upon what previous agents have said and provide analysis."""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    context = (
        f"This is round {round_count + 1}. "
        "Previous agents have shared their thoughts."
    )
    messages.append(HumanMessage(content=context))

    response = llm.invoke(messages)

    return {
        "messages": [response],
        "current_agent": "agent_c",
        "round_count": round_count + 1
    }

In [ ]:
# 6.4 · Agent C
def agent_c(state: NetworkState) -> NetworkState:
    messages = state["messages"]
    round_count = state.get("round_count", 0)

    system_prompt = """You are Agent C, a synthesis specialist.
You synthesize information from all agents and provide a comprehensive summary.
Combine insights from previous agents and add final thoughts."""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    context = f"This is round {round_count + 1}. Synthesize all previous contributions."
    messages.append(HumanMessage(content=context))

    response = llm.invoke(messages)

    if round_count >= 2:
        next_agent = "end"
    else:
        next_agent = "agent_a"

    return {
        "messages": [response],
        "current_agent": next_agent,
        "round_count": round_count + 1
    }

print("✅ Network agents created!")

In [ ]:
# 6.5 · Who comes next?
def route_network(state: NetworkState) -> Literal[
    "agent_a", "agent_b", "agent_c", "__end__"
]:
    current_agent = state.get("current_agent", "agent_a")
    round_count = state.get("round_count", 0)

    if current_agent == "end" or round_count >= 3:
        return "__end__"

    return current_agent

In [ ]:
# 6.6 · The graph
network_workflow = StateGraph(NetworkState)

network_workflow.add_node("agent_a", agent_a)
network_workflow.add_node("agent_b", agent_b)
network_workflow.add_node("agent_c", agent_c)

network_workflow.set_entry_point("agent_a")

network_workflow.add_conditional_edges(
    "agent_a",
    lambda state: "agent_b",
    {"agent_b": "agent_b"}
)

network_workflow.add_conditional_edges(
    "agent_b",
    lambda state: "agent_c",
    {"agent_c": "agent_c"}
)

network_workflow.add_conditional_edges(
    "agent_c",
    route_network,
    {
        "agent_a": "agent_a",
        "__end__": END
    }
)

network_app = network_workflow.compile()

print("✅ Network/swarm system built!")
print("\nNetwork flow: agent_a → agent_b → agent_c → (loop or end)")

In [ ]:
# 6.7 · A question for the whole table
print("=" * 60)
print("NETWORK SYSTEM TEST")
print("=" * 60)

network_query = "What are the key considerations for building a scalable web application?"
initial_state = {
    "messages": [HumanMessage(content=network_query)],
    "current_agent": "agent_a",
    "round_count": 0
}

result = network_app.invoke(initial_state)

print(f"\nUser Query: {network_query}")
print(f"""\nNumber of agent responses: {len([
    m for m in result['messages'] if isinstance(m, AIMessage)
])}""")
print(f"\nFinal Round Count: {result['round_count']}")
print("\n" + "=" * 60)
print("AGENT RESPONSES:")
print("=" * 60)

for i, msg in enumerate(result["messages"]):
    if isinstance(msg, AIMessage):
        print(f"\n--- Agent Response {i//2 + 1} ---")
        print(msg.text[:300] + "..." if len(msg.text) > 300 else msg.text)

#### 6.8 · What did we just see?

<details>
<summary>Click to show</summary>

**A fixed chain: A, then B, then C.**

The counter goes up with every agent. After A, B and C it is already 3, and the run ends.

In a real network, each agent would decide by itself who works next.

</details>

<div style="height:44px"></div>

## Part 7 of 8 · Do you need more than one agent?

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**Do you really need more than one agent?**

#### 7.1 · How can both be right?

<details>
<summary>Click to show</summary>

**Research** splits nicely: ten agents can read ten different things.

**Building one thing** does not split so well: the parts must fit together.

**A good rule: start with one agent. Add a second one only when you can say exactly why.**

---

From the long list in your own notebook: **one agent, one clear job. Limit the rounds. Test each agent alone first.**

</details>

<div style="height:44px"></div>

## Part 8 of 8 · Wrap-up of the unit and of the course

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span></div>

**What do you take with you?**

#### 8.1 · Five things from today

**1.** A multi-agent system is several agents, and often they are one model with different prompts.

**2.** A job title in a prompt is not a skill. A specialist also needs tools.

**3.** With tool calling, the manager gets a report. With a handoff, the call is transferred.

**4.** Decide who knows what: only one query, or the whole conversation.

**5.** A loop needs a limit.

#### 8.2 · The whole course

**Unit 3:** the model remembers nothing, so we give it everything, and a graph sets the steps.

**Unit 4:** about you, its job and today, the model only knows what is in the prompt.

**Unit 5:** the model never runs a tool. It only asks.

**Unit 6:** MCP is one plug for many tools.

**Unit 7:** you cannot fix what you cannot see.

**Unit 8:** RAG and CAG put knowledge into the prompt.

**Unit 9:** a picture is just more input.

**Unit 10:** many agents, but the hard part is who acts next and who knows what.

---

**One idea, again and again: the model is a very smart person in a closed room.** Almost everything that we built decides what goes under the door, and what we do with the note that comes back.

#### 8.3 · Your next step: the team assignment

Please read `team_assignment/assignment_description.md` carefully.

**An agent system** for a real problem, in a shared repository, each with your own Git account and your own commits. Each of you contributes to both the code and the report, and Git must show it.

**A technical report.** The template is `team_assignment/report.md`.

**A presentation.** Each of you must answer questions about your own part.

**You are evaluated on** the system, the report, your contributions in Git, and the presentation.

**Start with one agent, but solve a real problem, not a toy example.**